# Bài học 09 - Mẫu thiết kế Siêu nhận thức


## Cài đặt

Notebook này minh họa mẫu thiết kế Metacognition sử dụng Microsoft Agent Framework.

**Yêu cầu trước:**
- Bản triển khai Azure OpenAI được cấu hình thông qua các biến môi trường
- Azure CLI đã được xác thực (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Siêu nhận thức là gì?

Siêu nhận thức là **suy nghĩ về suy nghĩ**. Trong bối cảnh các tác nhân AI, nó có nghĩa là xây dựng các tác nhân có thể:

- **Tự suy ngẫm** về các kết quả đầu ra và quá trình lập luận của chính chúng
- **Phát hiện lỗi** và phục hồi một cách trơn tru thay vì thất bại một cách im lặng
- **Đánh giá** xem liệu các phản hồi của chúng có đầy đủ và hữu ích hay không
- **Thích nghi** chiến lược khi phương án ban đầu không hiệu quả (ví dụ: chuyển sang hệ thống dự phòng)

Một tác nhân siêu nhận thức không chỉ trả lời câu hỏi — nó giám sát hiệu suất của chính nó và điều chỉnh trong quá trình hoạt động.


## Công cụ chính và dự phòng

Một mẫu siêu nhận thức phổ biến là **chiến lược dự phòng**. Tác nhân thử dùng công cụ chính trước; nếu nó thất bại (ví dụ: lỗi 404), tác nhân nhận ra lỗi và minh bạch chuyển sang công cụ dự phòng.

Điều này phản ánh các hệ thống trong thế giới thực, nơi các dịch vụ chính có thể không khả dụng và các tác nhân phải tự chẩn đoán sự cố trước khi chọn một con đường thay thế.

Dưới đây chúng tôi định nghĩa hai công cụ tra cứu chuyến bay:
- **Chính** — bao gồm Paris, Tokyo và Barcelona
- **Dự phòng** — bao gồm Berlin, Sydney và New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Tác nhân tự phản chiếu với phục hồi lỗi

Tác nhân dưới đây được chỉ dẫn thử hệ thống bay chính trước, nhận biết các lỗi, và một cách minh bạch chuyển sang hệ thống dự phòng. Sau mỗi phản hồi, nó tự đánh giá ngắn gọn liệu có trả lời đầy đủ câu hỏi của người dùng hay không.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Mẫu Tự Đánh Giá

Một khía cạnh khác của siêu nhận thức là **tự đánh giá**: một tác nhân riêng (hoặc cùng tác nhân trong một lần chạy thứ hai) xem xét một phản hồi về độ đầy đủ, độ chính xác và tính hữu ích.

Dưới đây chúng ta tạo một tác nhân `ResponseEvaluator` chấm điểm các phản hồi của travel-agent theo ba khía cạnh.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Tóm tắt

Trong bài học này, bạn đã học cách xây dựng **tác nhân siêu nhận thức** sử dụng Microsoft Agent Framework:

- **Tự suy ngẫm**: Các tác nhân theo dõi chính quá trình lập luận của bản thân và truyền đạt một cách minh bạch những gì đã xảy ra.
- **Phục hồi lỗi với phương án dự phòng**: Một mô hình công cụ chính + dự phòng nơi tác nhân phát hiện lỗi (ví dụ: lỗi 404) và tự động thử nguồn thay thế.
- **Tự đánh giá**: Một tác nhân đánh giá riêng biệt chấm điểm các phản hồi về tính đầy đủ, độ chính xác và tính hữu ích.

Những mô hình này giúp các tác nhân trở nên mạnh mẽ hơn, minh bạch hơn và đáng tin cậy hơn — những phẩm chất quan trọng cho việc triển khai vào môi trường sản xuất.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Miễn trừ trách nhiệm:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI Co-op Translator (https://github.com/Azure/co-op-translator). Mặc dù chúng tôi nỗ lực đảm bảo độ chính xác, xin lưu ý rằng các bản dịch tự động có thể chứa lỗi hoặc thông tin không chính xác. Văn bản gốc bằng ngôn ngữ ban đầu nên được coi là nguồn tham chiếu chính thức. Đối với thông tin quan trọng, khuyến nghị sử dụng dịch vụ dịch thuật chuyên nghiệp do người thực hiện. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc diễn giải sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
